


# **Data Engineeer Sandbox**


## **Data Connection to Drive Folders**



In [1]:
# Mount Google Drive to access the data files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

# Define the base path where the data is stored (adjust this path)
DATA_PATH = '/content/drive/MyDrive/sdg_bc_sandbox/'

print("Setup complete. Data path is:", DATA_PATH)

# Install DuckDB for simple, high-performance in-memory/file-based SQL (optional but recommended)
!pip install duckdb
print("\n--- Setup Complete ---")

Setup complete. Data path is: /content/drive/MyDrive/sdg_bc_sandbox/

--- Setup Complete ---


In [3]:
import os

# List contents of the DATA_PATH
print(f"Listing contents of {DATA_PATH}:")
for root, dirs, files in os.walk(DATA_PATH):
    for file in files:
        print(os.path.join(root, file))
    for dir in dirs:
        print(os.path.join(root, dir))

Listing contents of /content/drive/MyDrive/sdg_bc_sandbox/:
/content/drive/MyDrive/sdg_bc_sandbox/Store_Territory.csv
/content/drive/MyDrive/sdg_bc_sandbox/Customer.csv
/content/drive/MyDrive/sdg_bc_sandbox/Currency.csv
/content/drive/MyDrive/sdg_bc_sandbox/Product_Sales_20221231.csv
/content/drive/MyDrive/sdg_bc_sandbox/Product_Sales_20210701.csv


## **Ingestion & Data Quality**

In [4]:
import pandas as pd
import duckdb

# Create a DuckDB connection
con = duckdb.connect()
version_duckdb = con.execute("SELECT version()").fetchone()[0]
print(f" DuckDB  initialised successfully (Version: {version_duckdb}).")

 DuckDB  initialised successfully (Version: v1.3.2).


### **Data ingestion (Bronze)**
This section handles the ingestion of raw files from Google Drive into DuckDB.
* The wildcard pattern `Product_Sales_*.csv` is used to automatically consolidate fragmented product sales files (`20210701` and `20221231`) into a single table with an optimised read operation.
* Data is loaded with no changes in structure and field names to keep data as raw as possible before to apply any business logic in the next data layer (silver).

In [5]:

# ==============================================================================
# Creation of raw staging tables
# ==============================================================================

# Load the dataframes:

try:
  #customer
  con.execute(f"""
          CREATE OR REPLACE TABLE customers_raw AS
          SELECT * FROM read_csv_auto('{DATA_PATH}Customer.csv', header=True);
      """)
  print("customers_raw loaded")
  #currency
  con.execute(f"""
          CREATE OR REPLACE TABLE currency_raw AS
          SELECT * FROM read_csv_auto('{DATA_PATH}Currency.csv', header=True);
      """)
  print("currency_raw loaded")
  #store_territory
  con.execute(f"""
          CREATE OR REPLACE TABLE store_territory_raw AS
          SELECT * FROM read_csv_auto('{DATA_PATH}Store_Territory.csv', header=True);
      """)
  print("Store_Territory loaded")

  # Read dinamically both product sales files to be unified into a single table
  con.execute(f"""
          CREATE OR REPLACE TABLE product_sales_raw AS
          SELECT * FROM read_csv_auto('{DATA_PATH}Product_Sales_*.csv', header=True);
      """)

  print("product_sales_raw loaded")

except Exception as e:
  print(f"Ingestion process failed: {str(e)}")


customers_raw loaded
currency_raw loaded
Store_Territory loaded


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

product_sales_raw loaded


In [6]:
# --- Verification Check ---
tablas = ['customers_raw', 'currency_raw', 'store_territory_raw', 'product_sales_raw']

for tabla in tablas:
    count = con.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    print(f"Table: {tabla} | Total Records: {count:,}")

Table: customers_raw | Total Records: 37,738
Table: currency_raw | Total Records: 28
Table: store_territory_raw | Total Records: 312
Table: product_sales_raw | Total Records: 365,256


In [13]:
test_query = con.sql("""
    SELECT *
    FROM fact_sales
    LIMIT 3;
""").df()
display(test_query)

,ID_Sales,ID_Channel,ID_Store,ID_Product,ID_Promotion,ID_Currency,Date,SalesQuantity,ReturnQuantity,ReturnAmount,DiscountQuantity,DiscountAmount,TotalCost,SalesAmount,NetQuantity,NetProfit,ProfitMargin
0,12,1,119,543,1,1,2020-04-29,10,0,0.0,0,0.0,1167.5,2290.0,10,1122.5,0.490175
1,53,1,121,709,1,1,2020-04-19,10,0,0.0,0,0.0,533.4,1160.0,10,626.6,0.540172
2,82,1,18,590,3,1,2020-07-10,12,0,0.0,2,199.8,5512.8,11788.2,12,6275.4,0.532346


## **Dimension Modeling & Core ELT**




### **Quality, Cleaning and Transformation (Silver)**

This section implements data cleaning and standardization rules to transform raw data into normalized datasets.

Four different methods will normalize data by removing duplicate columns and duplicate data bases on primary keys from every table, handling missing values by filling null values with expected values dependig on value types and casting data types to get data in a valid format to be used.

The last method will load normalized data into new tables to get a standarized data layer

In [8]:
# ==============================================================================
# Normalization methods
# ==============================================================================

import pandas as pd
import duckdb

def handle_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Scans the DataFrame for columns auto-suffixed with '_1'
    Compares data contents with the original base column, if contents match exactly, the redundant column is dropped.
    """
    columns_to_drop = []

    for col in df.columns:
        if col.startswith("EmployeeKey"):
            columns_to_drop.append(col)

    for col in df.columns:
        if col.endswith('_1'):
            base_col = col[:-2]
            if base_col in df.columns:
                if df[base_col].equals(df[col]):
                    columns_to_drop.append(col)

    # Execute drops
    if columns_to_drop:
        df = df.drop(columns=columns_to_drop)
        print(f"  Removed exact duplicate columns: {columns_to_drop}")

    return df

def remove_duplicates(df: pd.DataFrame, primary_key: str, order_by_col: str = None) -> pd.DataFrame:
    """
    Removes duplicate records based on a primary key.
    If an order_by_col is provided, it keeps the latest record.
    """
    if order_by_col and order_by_col in df.columns:
        # Convert to datetime temporarily to ensure accurate sorting sequence
        temp_date = pd.to_datetime(df[order_by_col], errors='coerce')
        df = df.iloc[temp_date.argsort()[::-1]]

    df_cleaned = df.drop_duplicates(subset=[primary_key], keep='first')
    return df_cleaned


def handle_missing_values(df: pd.DataFrame, string_fill: str = 'Unknown', numeric_fill: float = 0.0) -> pd.DataFrame:
    """
    Inspects columns and handles missing values based on data types.
    """
    for col in df.columns:
        # Check if the column is categorical/object text
        if df[col].dtype == 'object' or isinstance(df[col].dtype, pd.CategoricalDtype):
            df[col] = df[col].fillna(string_fill)
        # Check if the column is numeric
        elif pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(numeric_fill)

    return df


def cast_data_types(df: pd.DataFrame, type_mapping: dict) -> pd.DataFrame:
    """
    Casts dataframe columns to specified data types.
    Supports specific handling for date format
    """
    for col, data_type in type_mapping.items():
        if col in df.columns:
            if data_type == 'date':
                df[col] = pd.to_datetime(df[col], errors='coerce').dt.date
            else:
                df[col] = df[col].astype(data_type)
    return df



In [9]:

# ==============================================================================
# Orchestration process to apply normalization methods and create cleansed layer
# ==============================================================================

def cleaning_process(con: duckdb.DuckDBPyConnection, source_table: str, target_table: str,
                        primary_key: str, order_by_col: str = None, type_mapping: dict = None):


    print(f"Processing transformations from: {source_table} to {target_table}")

    # Extract data from DB raw staging layer to dataframes
    df = con.execute(f"SELECT * FROM {source_table}").df()

    # Apply standardisation methods
    df = handle_duplicate_columns(df)
    df = remove_duplicates(df, primary_key=primary_key, order_by_col=order_by_col)
    df = handle_missing_values(df)
    df = cast_data_types(df, type_mapping)

    # Load processed dataframe back into DB cleansed layer
    con.register('tmp_df', df)
    con.execute(f"CREATE OR REPLACE TABLE {target_table} AS SELECT * FROM tmp_df")
    con.unregister('tmp_df')

    print(f"Successfully loaded '{target_table}'")



In [10]:


try:
    # Schema mappings for each table
    customer_types = {
        'CustomerKey': 'int', 'GeographyKey': 'int', 'FirstName': 'str', 'LastName': 'str','BirthDate': 'date', 'MaritalStatus': 'str', 'Gender': 'str', 'YearlyIncome': 'float',
        'TotalChildren': 'int', 'NumberChildrenAtHome': 'int', 'Education': 'str','Occupation': 'str', 'HouseOwnerFlag': 'int', 'NumberCarsOwned': 'int', 'modified_date': 'date'
    }

    currency_types = {
      'CurrencyKey': 'int', 'CurrencyName': 'str', 'CurrencyDescription': 'str'
    }

    store_types = {
      'StoreKey': 'int', 'GeographyKey': 'int', 'StoreManager': 'str', 'StoreType': 'str','StoreName': 'str', 'Status': 'str', 'OpenDate': 'date', 'CloseDate': 'date',
      'EntityKey': 'int', 'StorePhone': 'str', 'StoreFax': 'str', 'CloseReason': 'str','EmployeeCount': 'int', 'SellingAreaSize': 'float', 'LastRemodelDate': 'date',
      'SalesTerritoryKey': 'int', 'SalesTerritoryName': 'str','SalesTerritoryRegion': 'str', 'SalesTerritoryCountry': 'str', 'SalesTerritoryGroup': 'str',
      'SalesTerritoryLevel': 'str', 'SalesTerritoryManager': 'int', 'Status_1': 'str'
    }

    sales_types = {
      'SalesKey': 'int', 'DateKey': 'date', 'channelKey': 'int', 'StoreKey': 'int', 'ProductKey': 'int', 'PromotionKey': 'int', 'CurrencyKey': 'int','UnitCost': 'float',
      'UnitPrice': 'float', 'SalesQuantity': 'int','ReturnQuantity': 'int', 'ReturnAmount': 'float', 'DiscountQuantity': 'int','DiscountAmount': 'float', 'TotalCost': 'float',
      'SalesAmount': 'float', 'ProductName': 'str', 'ProductDescription': 'str','ProductSubcategoryKey': 'int', 'Manufacturer': 'str', 'BrandName': 'str', 'ClassID': 'str',
      'ClassName': 'str', 'StyleID': 'int', 'StyleName': 'str','ColorID': 'int', 'ColorName': 'str', 'Weight': 'float', 'WeightUnitMeasureID': 'str', 'UnitOfMeasureID': 'int',
      'UnitOfMeasureName': 'str', 'StockTypeID': 'int', 'StockTypeName': 'str', 'AvailableForSaleDate': 'date', 'Status': 'str'
    }

    # Execute orchestrator function across all target datasets
    cleaning_process(con, 'customers_raw', 'customers_cleansed',
                        primary_key='CustomerKey', order_by_col='modified_date', type_mapping=customer_types)

    cleaning_process(con, 'currency_raw', 'currency_cleansed',
                        primary_key='CurrencyKey', type_mapping=currency_types)

    cleaning_process(con, 'store_territory_raw', 'store_territory_cleansed',
                        primary_key='StoreKey', type_mapping=store_types)

    cleaning_process(con, 'product_sales_raw', 'product_sales_cleansed',
                        primary_key='SalesKey', type_mapping=sales_types)

    print("All cleansed tables have been processed and loaded")

except Exception as e:
    print(f"❌ Pipeline failed: {str(e)}")


Processing transformations from: customers_raw to customers_cleansed
Successfully loaded 'customers_cleansed'
Processing transformations from: currency_raw to currency_cleansed
Successfully loaded 'currency_cleansed'
Processing transformations from: store_territory_raw to store_territory_cleansed
  Removed exact duplicate columns: ['EmployeeKey', 'EmployeeKey_1']
Successfully loaded 'store_territory_cleansed'
Processing transformations from: product_sales_raw to product_sales_cleansed
  Removed exact duplicate columns: ['ProductKey_1', 'UnitCost_1', 'UnitPrice_1']
Successfully loaded 'product_sales_cleansed'
All cleansed tables have been processed and loaded


In [18]:
# Validation check
cleansed_tables = [
    'customers_cleansed',
    'currency_cleansed',
    'store_territory_cleansed',
    'product_sales_cleansed'
]

for table in cleansed_tables:
    # Get row count
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    # Get column count
    col_count = len(con.execute(f"SELECT * FROM {table} LIMIT 1").df().columns)

    print(f"{table} | Columns: {col_count} | Total Records: {count:,}")

customers_cleansed | Columns: 15 | Total Records: 18,869
currency_cleansed | Columns: 3 | Total Records: 28
store_territory_cleansed | Columns: 23 | Total Records: 306
product_sales_cleansed | Columns: 35 | Total Records: 208,201


### **Dimensional Modeling (Gold)**

This final section will create an star schema model based on cleansed tables with some metrics created to be used in BI tools

As data product and sales product are in the same table `product_sales_cleansed`, to avoid data redundancy, the strategy is create a dimensional table with product data.
In order to improve analytical information in the model, new KPIs are computed at the row level within the `fact_sales` table: `NetQuantity` calculated as `SalesQuantity - ReturnQuantity`, `NetProfit` calculated as `SalesAmount - TotalCost`and `ProfitMargin` as ratio of Net Profit to total Sales Amount.

The Final Star Schema Entities will be `dim_customer`, `dim_currency`, `dim_store_territory`, `dim_product`, and `fact_sales`.

In [12]:

try:

    # Dimension Customer

    con.execute("""
        CREATE TABLE dim_customer AS
        SELECT
            CustomerKey AS ID_Customer,
            GeographyKey AS ID_Geography,
            FirstName,
            LastName,
            BirthDate,
            MaritalStatus,
            Gender,
            YearlyIncome
        FROM customers_cleansed;
    """)

    print("Dimension created: 'dim_customer'")


    # Dimension Currency

    con.execute("""
        CREATE OR REPLACE TABLE dim_currency AS
        SELECT
        CurrencyKey as ID_Currency,
        CurrencyName,
        CurrencyDescription
        FROM currency_cleansed;
    """)
    print("Dimension created: 'dim_currency'")


    # Dimension Store territory

    con.execute("""
        CREATE OR REPLACE TABLE dim_store_territory AS
        SELECT
            StoreKey AS ID_Store,
            StoreManager AS ID_StoreManager,
            SalesTerritoryKey AS ID_SalesTerritory,
            SalesTerritoryManager AS ID_TerritoryManager,
            EntityKey AS ID_Entity,
            StoreName,
            StoreType,
            Status,
            OpenDate,
            CloseDate,
            EmployeeCount,
            SalesTerritoryName,
            SalesTerritoryRegion,
            SalesTerritoryCountry,
            SalesTerritoryGroup
        FROM store_territory_cleansed;
    """)
    print("Dimension created: 'dim_store_territory'")


    # Dimension Product

    con.execute("""
        CREATE OR REPLACE TABLE dim_product AS
        SELECT DISTINCT
            ProductKey AS ID_Product,
            ProductSubcategoryKey AS ID_ProductSubcategory,
            ProductName,
            ProductDescription,
            Manufacturer,
            BrandName,
            ClassName,
            UnitCost,
            UnitPrice,
            Status AS ProductStatus,
            AvailableForSaleDate
        FROM product_sales_cleansed
        WHERE ProductKey IS NOT NULL;
    """)
    print("Dimension created: 'dim_product'")


    # Fact table Sales

    con.execute("""
        CREATE OR REPLACE TABLE fact_sales AS
        SELECT
            SalesKey AS ID_Sales,
            channelKey AS ID_Channel,
            StoreKey AS ID_Store,
            ProductKey AS ID_Product,
            PromotionKey AS ID_Promotion,
            CurrencyKey AS ID_Currency,
            DateKey AS Date,
            SalesQuantity,
            ReturnQuantity,
            ReturnAmount,
            DiscountQuantity,
            DiscountAmount,
            TotalCost,
            SalesAmount,
            -- Tailored KPIs
            (SalesQuantity - ReturnQuantity) AS NetQuantity,
            (SalesAmount - TotalCost) AS NetProfit,
            CASE
                WHEN SalesAmount > 0 THEN (SalesAmount - TotalCost) / SalesAmount
                ELSE 0
            END AS ProfitMargin

        FROM product_sales_cleansed;
    """)
    print("Fact table created: 'fact_sales'")

    print(" Dimensional Model completed")

except Exception as e:
    print(f"Error during Dimensional Model creation: {str(e)}")

Dimension created: 'dim_customer'
Dimension created: 'dim_currency'
Dimension created: 'dim_store_territory'
Dimension created: 'dim_product'
Fact table created: 'fact_sales'
 Dimensional Model completed


In [16]:
# Validation check

dimensional_tables = [
    'dim_customer',
    'dim_currency',
    'dim_store_territory',
    'dim_product',
    'fact_sales'
]

for table in dimensional_tables:
    # Get row count
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    # Get column count
    col_count = len(con.execute(f"SELECT * FROM {table} LIMIT 1").df().columns)

    print(f"{table} | Columns: {col_count} | Total Records: {count:,}")

print("Tailored KPIs Preview")
preview_kpis = con.execute("""
    SELECT ID_Sales, SalesAmount, TotalCost, NetProfit, ROUND(ProfitMargin, 2) as Margin
    FROM fact_sales
    WHERE NetProfit > 0
    LIMIT 3
""").df()
display(preview_kpis)

dim_customer | Columns: 8 | Total Records: 18,869
dim_currency | Columns: 3 | Total Records: 28
dim_store_territory | Columns: 15 | Total Records: 306
dim_product | Columns: 11 | Total Records: 718
fact_sales | Columns: 17 | Total Records: 208,201
Tailored KPIs Preview


,ID_Sales,SalesAmount,TotalCost,NetProfit,Margin
0,12,2290.0,1167.5,1122.5,0.49
1,53,1160.0,533.4,626.6,0.54
2,82,11788.2,5512.8,6275.4,0.53
